# Colab runner — Semarang Batik (Kelompok 5, Track A)

Runs the frozen experiment matrix on a GPU. The split manifest committed in `data/manifests/split.csv` is **reused, never regenerated**, so results are comparable everywhere.

Runtime → Change runtime type → GPU. Runs (`artifacts/runs`) and the resized image cache live on Google Drive, so after a disconnect re-run all cells: finished runs are skipped, an interrupted run resumes from `last.pt`, and the 13 GB download is not repeated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/Universitas-Cakrawala/semarang-batik-fine-grained-transfer-learning.git'
BRANCH = 'main'
# Reused runtime: pull, otherwise Colab keeps running stale code
!if [ -d /content/repo ]; then git -C /content/repo pull --ff-only; else git clone -b $BRANCH $REPO_URL /content/repo; fi
%cd /content/repo
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Persist runs and the image cache on Drive (symlinks) so a disconnect never loses work
DRIVE = '/content/drive/MyDrive/batik'
!mkdir -p $DRIVE/runs $DRIVE/processed data
!rm -rf artifacts/runs data/processed && ln -s $DRIVE/runs artifacts/runs && ln -s $DRIVE/processed data/processed

In [ ]:
# One-off: ~13 GB download + cache build (EXIF-transposed, short side 320 px). Skipped when the cache exists.
import pathlib
if not any(pathlib.Path('data/processed/short320').glob('*/*.jpg')):
    !python -m batik download --delete-zip
    !python -m batik audit --workers 2
    !git diff --stat data/manifests   # expected: no changes (audit is deterministic)
!python -m batik verify-split   # must print OK: the committed split is used as-is

In [ ]:
!python -m batik sanity

In [ ]:
# G1-G3 x 3 seeds, G4, G1g-G3g, then G5. Re-run this cell after a disconnect to resume.
!python -m batik matrix --num-workers 2

In [ ]:
!python -m batik aggregate
!python -m batik robustness
!python -m batik errors
!python -m batik release
!cat artifacts/metrics/tables.md
# Keep the small, committable outputs on Drive as well
!mkdir -p $DRIVE/outputs && cp -r artifacts/metrics artifacts/error_analysis artifacts/release figures $DRIVE/outputs/